[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sistemas-Multimedia/VCF/blob/main/notebooks/2D-LBT.ipynb)

## LBT (Learning Block Transformation) image compression

# [VCF](https://github.com/Sistemas-Multimedia/VCF)
## [2D-LBT](https://sistemas-multimedia.github.io/contents/spatial_transforms/#rdoptimized-learned-block-transform-lbt)

Image compression using the 2D-LBT.

In [ ]:
%run download_VCF.ipynb

In [ ]:
from IPython.display import Image, display

In [ ]:
%run download_default_image.ipynb

In [ ]:
%%bash
python ../src/2D-LBT.py encode -h

### Tamaño de bloque

La imagen se divide en bloques de $B \times B$ píxeles. Cada bloque se procesa de forma independiente:

- **Bloques pequeños** (ej. 4×4): Capturan detalles finos pero tienen menos capacidad de compactación de energía
- **Bloques grandes** (ej. 16×16): Mayor capacidad de compactación pero pueden generar artefactos de bloque más visibles

La función `_extract_blocks()` divide la imagen en bloques y los aplana a vectores de $B^2$ elementos. La función `_reconstruct_map()` realiza la operación inversa para reconstruir la imagen.

In [ ]:
%%bash
rm /tmp/encoded*
python ../src/2D-LBT.py encode -q 1 --block_size 4
rm /tmp/decoded.png
python ../src/2D-LBT.py decode -q 1 --block_size 4
python ../src/RDE.py

In [ ]:
display(Image(filename="/tmp/decoded.png"))

In [ ]:
%%bash
rm /tmp/encoded*
python ../src/2D-LBT.py encode -q 1 --block_size 16
rm /tmp/decoded.png
python ../src/2D-LBT.py decode -q 1 --block_size 16
python ../src/RDE.py

In [ ]:
display(Image(filename="/tmp/decoded.png"))

In [ ]:
%%bash 
rm /tmp/encoded*
python ../src/2D-LBT.py encode -q 1 --block_size 4 --epochs 10 --lr 1e-2
rm /tmp/decoded.png
python ../src/2D-LBT.py decode -q 1 --block_size 4
python ../src/RDE.py

In [ ]:
display(Image(filename="/tmp/decoded.png"))

In [ ]:
%%bash
rm /tmp/encoded*
python ../src/2D-LBT.py encode -q 1 --block_size 8 --epochs 10 --lr 1e-2 --L 10
rm /tmp/decoded.png
python ../src/2D-LBT.py decode -q 1 --block_size 8
python ../src/RDE.py

In [ ]:
display(Image(filename="/tmp/decoded.png"))

### Quantization

La cuantización es el proceso de reducir la precisión de los coeficientes transformados para lograr compresión. Consiste en dividir cada coeficiente por un paso de cuantización (`-q`) y redondear al entero más cercano.

- **Mayor `-q`**: Más compresión pero mayor pérdida de calidad (más artefactos)
- **Menor `-q`**: Mejor calidad pero menor compresión

**Cuantización perceptual** (`-p`): Utiliza las matrices de cuantización de JPEG, que aplican cuantización más agresiva a las frecuencias altas (menos perceptibles por el ojo humano) y más suave a las frecuencias bajas (más importantes visualmente). Las matrices `Y_QSSs` y `C_QSSs` definen los pasos de cuantización para luminancia y crominancia respectivamente.

In [ ]:
%%bash
rm /tmp/encoded*
python ../src/2D-LBT.py encode -q 32 --block_size 16
rm /tmp/decoded.png
python ../src/2D-LBT.py decode -q 32 --block_size 16
python ../src/RDE.py

In [ ]:
display(Image(filename="/tmp/decoded.png"))

In [ ]:
%%bash
rm /tmp/encoded*
python ../src/2D-LBT.py encode -p -q 8 -B 32
rm /tmp/decoded.png
python ../src/2D-LBT.py decode -p -q 8 -B 32
python ../src/RDE.py

In [ ]:
display(Image(filename="/tmp/decoded.png"))

## Using LloydMax

**LloydMax** es un cuantizador no uniforme que optimiza los niveles de cuantización según la distribución estadística de los datos. A diferencia del cuantizador uniforme (deadzone), LloydMax:

- Asigna más niveles de cuantización donde hay mayor densidad de valores
- Minimiza el error cuadrático medio de cuantización
- Requiere especificar el rango de valores esperados con `-m` (mínimo) y `-n` (máximo)

Es especialmente útil cuando la distribución de los coeficientes no es uniforme, lo cual es típico en transformadas como DCT o LBT.

In [ ]:
%%bash
rm /tmp/encoded*
python ../src/2D-LBT.py encode -q 16 -a LloydMax -m -2048 -n 2047 
rm /tmp/decoded.png
python ../src/2D-LBT.py decode -q 16 -a LloydMax -m -2048 -n 2047 
python ../src/RDE.py

In [ ]:
display(Image(filename="/tmp/decoded.png"))

In [ ]:
%%bash
rm /tmp/encoded*
python ../src/2D-LBT.py encode -q 1 -a LloydMax -m -2048 -n 2047 
rm /tmp/decoded.png
python ../src/2D-LBT.py decode -q 1 -a LloydMax -m -2048 -n 2047 
python ../src/RDE.py

In [ ]:
display(Image(filename="/tmp/decoded.png"))